____
### 1. Introduction to autoregression
- predicting the next value of a sequence based on what came before

$$x_t = f(x_{t-1}, x_{t-2}, ..., x_{t-k})$$
- the sequence itself ($x_{t-1}, x_{t-2}, ..., x_{t-k}$) supplies both the `input` and the `target`
- the seq is sliced into a window of past values (`X`) and the value that immediately follows is the output (`Y`)
- every point in the sequence becomes after the first few becomes a training example

#### 1.1 Autoregression as supervised learning
- Supervised learning requires 
    1. Input features `X`
    2. Ground truth targets `y`
    3. Loss function to measure the gap between predictions and targets `Loss fn`
- Autoregression contains
    1. window of past tokens/values (`X`)
    2. actual next token generated, known as it is within the dataset (`y`)
    3. cross-entropy (for discrete tokens) or MSE/MAE (continuous values) , comparing the model's predicted next value to the actual one in the dataset (`Loss fn`)

#### 1.2 Creating dataset for training
- given a large dataset, training examples are built using **sliding window**
- for a sequence `[x1, x2, x3, x4, x5, ...]`

| Input (X)              | Target (y) |
|-------------------------|------------|
| x1, x2, x3               | x4         |
| x2, x3, x4               | x5         |
| x3, x4, x5               | x6         |

- there are 2 common windowing strategies: **Fixed context window** and **Causal masking**
1. Fixed context window (`sliding window`)
    - always use the last `k` steps to predict the next one
    - when progressoing forward by 1 step, the last step is discarded and no longer used as the 
2. Causal masking (over the full sequence)
    - feeding the whole sequence at once
    - masks future positions so each position only attends to earlier ones
    - how transformer-based language models train
    - one forward pass generates predictions for EVERY position in the sequence simultaneously
    - each treated as its own supervised example

#### 1.3 Training process
1. Prepare the dataset
    - chunk sequence(s) into `(X, y)` pairs using the windowing method
    - Shuffle at the sequence level (not within a sequence — order inside a window must stay intact).
2. Forward pass: 
    - feed `X` into the model to receive a output
    - output generated takes 2 forms, depending on the datatype of the token
    - discrete - probability distribution over the next token
    (classification-style, via softmax)
    - continuous - a continuous value (regression-style).

3. Compute loss
    - compare prediction generated to the real y.
    - **cross-entropy loss** for discrete/categorical tgts
    - **MSE / MAE** for continuous targets 

4. Backpropagation
    - based on the loss function used, compute gradients of the loss with respect to every model parameter.

5. Optimizer step
    - update weights of the model to minimise loss

6. Repeat **steps 2-5** 
    - repeat over batches and epochs until the loss converges on a held-out validation set




#### 1.4 Teacher forcing
- During **training**, the model is fed the TRUE previous values as input instead of its own generated predictions that could be wrong
- This keeps training stable and parallelizable, as the training for each step is done using correct data
- If it were to feed its own output in during training, errors made would be passed forward into the next step, causing the error to snowball

During **inference**, this is switched off and the model has no ground truth to lean on
- it feeds its own previous outputs back in as the next input
- therefore autoregression generation is sequential (waits for the previous one to get the next) even though training can be parallel

#### Language model example:
Sequence: `"HELLO"`
With a context window of 3:
| Input     | Target |
|-----------|--------|
| `H E L`   | `L`    |
| `E L L`   | `O`    |

- Each row is one supervised training example. 
- Loss = cross-entropy between the model's predicted next-character distribution and the actual next character.

____
### 2. Setting context for SQL autocomplete model
- model reads the currently built sequence of SQL blocks to suggest the next block
- data is set from a fixed problem bank with 3020 common SQL patterns from a fixed problem bank
- the answer at each step is essentially deterministic, with only a few possible answers being the correct one

#### 2.1 Components of the autoregression problem
1. Tokens
    - the tokens in this problem is a discrete set from the pre-determined list of tokens that are supported by the web app

In [ ]:
VERTICAL_BLOCKS = [
    "SELECT",
    "FROM",
    "WHERE",
    "GROUP_BY",
    "HAVING",
    "ORDER_BY",
    "LIMIT",
    "JOIN",
]

HORIZONTAL_BLOCKS = [
    "DISTINCT",
    "COLUMN",           # <column name>, optionally alias-qualified
    "STAR",              # (*)
    "TABLE",              # <table name>, optionally aliased
    "IS_NULL",
    "IS_NOT_NULL",
    "OPERATOR",           # dropdown: =, >=, <=, <, >, !=, <>
    "LIKE",
    "IN",
    "NOT",
    "AND",
    "OR",
    "VALUE",
    "AGG_FUNC",           # dropdown: COUNT, SUM, AVG, MIN, MAX -- wraps COLUMN/STAR
    "ASC",
    "DESC",
    "OFFSET",
    "ON",
    "SUBQUERY_START",     # opens a nested query -- pushes a new frame
    "SUBQUERY_END",       # closes it -- placed after the nested frame auto-pops
]


2. Input `X`
    - the query that is currently built, a sequence of tokens `SELECT,COLUMN, FROM, TABLE, WHERE, ...`
3. Output `Y`
    - the predicted next output given the whole sequence that is currently built
    - follows the formula 
    $$y = x_t = f(x_{t-1}, x_{t-2}, ..., x_{t-k})$$